# SQL Business Analysis (Olist)
**Project:** E-Commerce Customer & Sales Analytics  
**Tools:** Google Colab + DuckDB  
**Goal:** Revenue definitions, join grain, monthly orders, AOV, and observed repeat customers.

Dataset source: [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

### Today's definitions
- **Completed orders:** `order_status = 'delivered'`.
- **Merchandise GMV:** sum of `order_items.price` associated with delivered orders, **excluding freight**. This is a merchandise-value proxy, **not net recognized accounting revenue or platform commission**.
- **Recorded payments:** sum of `order_payments.payment_value`. Includes effects like freight/payment composition; it is **not interchangeable with merchandise GMV**.
- **AOV:** merchandise GMV divided by number of delivered orders.
- **Repeat customer rate:** share of `customer_unique_id` with >1 delivered order in the observed dataset; not lifetime retention.

## 1 — Setup DuckDB

In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'duckdb'])
import duckdb
from pathlib import Path
from IPython.display import display

con = duckdb.connect(database=':memory:')
print('DuckDB version:', duckdb.__version__)

DuckDB version: 1.3.2


## 2 — Upload Kaggle ZIP


In [2]:
from pathlib import Path
import io, zipfile

DATA_DIR = Path('/content/olist_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
required = [
    'olist_orders_dataset.csv',
    'olist_customers_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv'
]
missing = [f for f in required if not (DATA_DIR / f).exists()]

if missing:
    from google.colab import files
    uploaded = files.upload()  # Select Kaggle ZIP or the 4 missing CSVs
    for name, raw in uploaded.items():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(raw)) as z:
                for member in z.infolist():
                    basename = Path(member.filename).name
                    if basename in required and not member.is_dir():
                        (DATA_DIR / basename).write_bytes(z.read(member))
        elif Path(name).name in required:
            (DATA_DIR / Path(name).name).write_bytes(raw)

missing = [f for f in required if not (DATA_DIR / f).exists()]
assert not missing, f'Missing files: {missing}. Upload those CSVs and rerun this cell.'
for name in required:
    print(f'✓ {name}: {(DATA_DIR / name).stat().st_size:,} bytes')

Saving archive.zip to archive.zip
✓ olist_orders_dataset.csv: 17,654,914 bytes
✓ olist_customers_dataset.csv: 9,033,957 bytes
✓ olist_order_items_dataset.csv: 15,438,671 bytes
✓ olist_order_payments_dataset.csv: 5,777,138 bytes


## 3 — Register CSV views in DuckDB
A *view* behaves like a SQL table, but points to the original CSV. No database server required.

In [3]:
sql_setup = r"""-- Raw CSV views: keep source files unchanged.
CREATE OR REPLACE VIEW orders AS
SELECT * FROM read_csv_auto('{DATA_DIR}/olist_orders_dataset.csv', header=true);
CREATE OR REPLACE VIEW customers AS
SELECT * FROM read_csv_auto('{DATA_DIR}/olist_customers_dataset.csv', header=true);
CREATE OR REPLACE VIEW order_items AS
SELECT * FROM read_csv_auto('{DATA_DIR}/olist_order_items_dataset.csv', header=true);
CREATE OR REPLACE VIEW order_payments AS
SELECT * FROM read_csv_auto('{DATA_DIR}/olist_order_payments_dataset.csv', header=true);

-- One row per order_id BEFORE joining to orders.
CREATE OR REPLACE VIEW v_item_totals AS
SELECT
    order_id,
    COUNT(*) AS item_rows,
    SUM(TRY_CAST(price AS DOUBLE)) AS merchandise_value,
    SUM(TRY_CAST(freight_value AS DOUBLE)) AS freight_value
FROM order_items
GROUP BY order_id;

CREATE OR REPLACE VIEW v_payment_totals AS
SELECT
    order_id,
    COUNT(*) AS payment_rows,
    SUM(TRY_CAST(payment_value AS DOUBLE)) AS payment_value
FROM order_payments
GROUP BY order_id;

-- Analysis grain: exactly one row per delivered order.
CREATE OR REPLACE VIEW v_order_sales AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    i.item_rows,
    i.merchandise_value,
    i.freight_value,
    p.payment_rows,
    p.payment_value
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN v_item_totals i ON o.order_id = i.order_id
LEFT JOIN v_payment_totals p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';"""
con.execute(sql_setup.replace('{DATA_DIR}', DATA_DIR.as_posix()))
print('Created raw views and 3 analytical views!')

Created raw views and 3 analytical views!


## 4 — Sanity check: total and delivered orders

In [4]:
sql = """SELECT
    COUNT(*) AS total_orders,
    COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered_orders,
    COUNT(DISTINCT order_id) AS unique_order_ids
FROM orders;"""
display(con.execute(sql).df())

,total_orders,delivered_orders,unique_order_ids
0,99441,96478,99441


### 5 — Understand the JOIN trap
`order_items` has multiple item rows per order, and `order_payments` can have multiple payment records per order.

In [5]:
display(con.execute("""SELECT
    (SELECT COUNT(*) FROM order_items) AS item_rows,
    (SELECT COUNT(DISTINCT order_id) FROM order_items) AS orders_with_items,
    (SELECT COUNT(*) FROM order_payments) AS payment_rows,
    (SELECT COUNT(DISTINCT order_id) FROM order_payments) AS orders_with_payments;""").df())
print('Final order-level preview:')
display(con.execute('SELECT * FROM v_order_sales LIMIT 5').df())

,item_rows,orders_with_items,payment_rows,orders_with_payments
0,112650,98666,103886,99440


Final order-level preview:


,order_id,customer_id,customer_unique_id,customer_state,order_purchase_timestamp,order_delivered_customer_date,item_rows,merchandise_value,freight_value,payment_rows,payment_value
0,00e7ee1b050b8499577073aeb2a297a1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,SP,2017-05-16 15:05:35,2017-05-25 10:35:35,1,124.99,21.88,1,146.87
1,29150127e6685892b6eab3eec79f59c7,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,SP,2018-01-12 20:48:24,2018-01-29 12:41:19,1,289.00,46.48,1,335.48
2,b2059ed67ce144a36e2aa97d2c9e9ad2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,SP,2018-05-19 16:07:45,2018-06-14 17:58:51,1,139.94,17.79,1,157.73
3,951670f92359f4fe4a63112aa7306eba,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,SP,2018-03-13 16:06:38,2018-03-28 16:04:25,1,149.94,23.36,1,173.30
4,6b7d50bd145f6fc7f33cebabd7e49d0f,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,SP,2018-07-29 09:51:30,2018-08-09 20:55:48,1,230.00,22.25,1,252.25


## 6 — Validate the final one-row-per-order view
**Expected:** delivered_rows = unique_orders = 96,478. Review any missing items / payments / customers rather than silently deleting rows.

In [6]:
validation = con.execute("""SELECT
    COUNT(*) AS delivered_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(*) FILTER (WHERE merchandise_value IS NULL) AS orders_missing_items,
    COUNT(*) FILTER (WHERE payment_value IS NULL) AS orders_missing_payments,
    COUNT(*) FILTER (WHERE customer_unique_id IS NULL) AS orders_missing_customer_id
FROM v_order_sales;""").df()
display(validation)
assert validation['delivered_rows'][0] == validation['unique_orders'][0], 'Order grain is duplicated!'
print('✓ Grain check passed: one row per delivered order')

,delivered_rows,unique_orders,orders_missing_items,orders_missing_payments,orders_missing_customer_id
0,96478,96478,0,1,0


✓ Grain check passed: one row per delivered order


## 7 — Calculate merchandise GMV, payments and AOV


In [7]:
display(con.execute("""SELECT
    COUNT(*) AS delivered_orders,
    ROUND(SUM(COALESCE(merchandise_value, 0)), 2) AS merchandise_gmv_brl,
    ROUND(SUM(COALESCE(freight_value, 0)), 2) AS shipping_charges_brl,
    ROUND(SUM(COALESCE(payment_value, 0)), 2) AS recorded_payments_brl,
    ROUND(SUM(COALESCE(merchandise_value, 0)) / NULLIF(COUNT(*), 0), 2) AS aov_merchandise_brl
FROM v_order_sales;""").df())

,delivered_orders,merchandise_gmv_brl,shipping_charges_brl,recorded_payments_brl,aov_merchandise_brl
0,96478,13221498.11,2198275.64,15422461.77,137.04


## 8 — Monthly sales trend
Month is based on **purchase date** for orders whose *final status* is delivered. This is not delivery-month revenue recognition.

In [8]:
monthly_sales = con.execute("""SELECT
    DATE_TRUNC('month', TRY_CAST(order_purchase_timestamp AS TIMESTAMP))::DATE AS purchase_month,
    COUNT(*) AS delivered_orders,
    ROUND(SUM(COALESCE(merchandise_value, 0)), 2) AS merchandise_gmv_brl,
    ROUND(SUM(COALESCE(merchandise_value, 0)) / NULLIF(COUNT(*), 0), 2) AS aov_brl
FROM v_order_sales
GROUP BY 1
ORDER BY 1;""").df()
display(monthly_sales.tail(12))
display(monthly_sales.loc[monthly_sales['purchase_month'].astype(str).str.startswith('2017-11')])

,purchase_month,delivered_orders,merchandise_gmv_brl,aov_brl
11,2017-09-01,4150,607399.67,146.36
12,2017-10-01,4478,648247.65,144.76
13,2017-11-01,7289,987765.37,135.51
14,2017-12-01,5513,726033.19,131.69
15,2018-01-01,7069,924645.00,130.80
16,2018-02-01,6555,826437.13,126.08
17,2018-03-01,7003,953356.25,136.14
18,2018-04-01,6798,973534.09,143.21
19,2018-05-01,6749,977544.69,144.84
20,2018-06-01,6099,856077.86,140.36


,purchase_month,delivered_orders,merchandise_gmv_brl,aov_brl
13,2017-11-01,7289,987765.37,135.51


## 9 — Observed repeat purchasing
Use `customer_unique_id`, not `customer_id`. Count **distinct delivered orders per customer**.

In [9]:
display(con.execute("""WITH customer_orders AS (
    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id) AS delivered_order_count
    FROM v_order_sales
    WHERE customer_unique_id IS NOT NULL
    GROUP BY 1
)
SELECT
    COUNT(*) AS unique_customers,
    COUNT(*) FILTER (WHERE delivered_order_count > 1) AS repeat_customers,
    ROUND(100.0 * COUNT(*) FILTER (WHERE delivered_order_count > 1)
        / NULLIF(COUNT(*), 0), 2) AS observed_repeat_rate_pct
FROM customer_orders;""").df())

,unique_customers,repeat_customers,observed_repeat_rate_pct
0,93358,2801,3.0


## 10 — Geographic merchandise GMV
`customer_state` is the buyer's state (not necessarily where the seller or warehouse is). Use this as **sales by customer location**.

In [10]:
sales_by_state = con.execute("""SELECT
    customer_state,
    COUNT(*) AS delivered_orders,
    ROUND(SUM(COALESCE(merchandise_value, 0)), 2) AS merchandise_gmv_brl,
    ROUND(AVG(merchandise_value), 2) AS avg_order_merchandise_value_brl
FROM v_order_sales
WHERE customer_state IS NOT NULL
GROUP BY 1
ORDER BY merchandise_gmv_brl DESC;""").df()
display(sales_by_state.head(10))

,customer_state,delivered_orders,merchandise_gmv_brl,avg_order_merchandise_value_brl
0,SP,40501,5067633.16,125.12
1,RJ,12350,1759651.13,142.48
2,MG,11354,1552481.83,136.73
3,RS,5345,728897.47,136.37
4,PR,4923,666063.51,135.30
5,SC,3546,507012.13,142.98
6,BA,3256,493584.14,151.59
7,DF,2080,296498.41,142.55
8,GO,1957,282836.70,144.53
9,ES,1995,268643.45,134.66


## 11 — Demonstrate why the wrong JOIN inflates totals


In [11]:
wrong = con.execute("""-- DEMONSTRATION ONLY: WRONG way to compute sales (can double-count).
SELECT
    ROUND(SUM(TRY_CAST(i.price AS DOUBLE)), 2) AS incorrect_merchandise_gmv_brl
FROM orders o
LEFT JOIN order_items i ON o.order_id = i.order_id
LEFT JOIN order_payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';""").df().iloc[0,0]
correct = con.execute("""SELECT ROUND(SUM(merchandise_value), 2) FROM v_order_sales""").fetchone()[0]
print('Incorrect direct-JOIN total (BRL):', wrong)
print('Correct order-level total (BRL):', correct)
print('Difference due to join multiplication (BRL):', round(float(wrong)-float(correct),2))

Incorrect direct-JOIN total (BRL): 13813963.68
Correct order-level total (BRL): 13221498.11
Difference due to join multiplication (BRL): 592465.57


## 12 — Export aggregate files for later Tableau work


In [12]:
EXPORT_DIR = Path('/content/olist_sql_exports')
EXPORT_DIR.mkdir(exist_ok=True, parents=True)
monthly_sales.to_csv(EXPORT_DIR / 'monthly_sales.csv', index=False)
sales_by_state.to_csv(EXPORT_DIR / 'sales_by_state.csv', index=False)
print('Exported:')
for p in sorted(EXPORT_DIR.glob('*.csv')):
    print('-', p)
# To download: uncomment next lines
# from google.colab import files
# files.download(str(EXPORT_DIR / 'monthly_sales.csv'))
# files.download(str(EXPORT_DIR / 'sales_by_state.csv'))

Exported:
- /content/olist_sql_exports/monthly_sales.csv
- /content/olist_sql_exports/sales_by_state.csv
